In [1]:
# instalar los conectores de LangChain y Google
!pip install langchain langchain-community chromadb sentence-transformers google-generativeai langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 84.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 72.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 9.1 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-exporter-otlp-proto-common==1.38.0, but you have opentelemetry-exporter-otlp-proto-common 1.43.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-proto==1.38.0, but you have opentelemetry-proto 1.43.0 which is incompatib

In [5]:
!pip install -U google-generativeai langchain-google-genai

In [9]:
import google.generativeai as genai
from google.colab import userdata

# 1. Configuramos la llave
mi_llave = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=mi_llave)

print("🔍 Consultando los servidores de Google para ver tus modelos disponibles...\n")

try:
    # 2. Le pedimos a Google la lista de modelos que soportan generación de texto
    modelos_disponibles = []
    for m in genai.list_models():
        if 'generateContent' in m.supported_generation_methods:
            print(f"✅ Habilitado: {m.name}")
            modelos_disponibles.append(m.name)

    if not modelos_disponibles:
        print("⚠️ Tu clave es válida, pero no tiene modelos de texto asignados.")
except Exception as e:
    print(f"❌ Error al intentar listar los modelos: {e}")

🔍 Consultando los servidores de Google para ver tus modelos disponibles...

✅ Habilitado: models/gemini-2.5-flash
✅ Habilitado: models/gemini-2.5-pro
✅ Habilitado: models/gemini-2.0-flash
✅ Habilitado: models/gemini-2.0-flash-001
✅ Habilitado: models/gemini-2.0-flash-lite-001
✅ Habilitado: models/gemini-2.0-flash-lite
✅ Habilitado: models/gemini-2.5-flash-preview-tts
✅ Habilitado: models/gemini-2.5-pro-preview-tts
✅ Habilitado: models/gemma-4-26b-a4b-it
✅ Habilitado: models/gemma-4-31b-it
✅ Habilitado: models/gemini-flash-latest
✅ Habilitado: models/gemini-flash-lite-latest
✅ Habilitado: models/gemini-pro-latest
✅ Habilitado: models/gemini-2.5-flash-lite
✅ Habilitado: models/gemini-2.5-flash-image
✅ Habilitado: models/gemini-3-pro-preview
✅ Habilitado: models/gemini-3-flash-preview
✅ Habilitado: models/gemini-3.1-pro-preview
✅ Habilitado: models/gemini-3.1-pro-preview-customtools
✅ Habilitado: models/gemini-3.1-flash-lite-preview
✅ Habilitado: models/gemini-3.1-flash-lite
✅ Habilitado:

In [10]:
import os
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. Recuperamos la llave
os.environ["GOOGLE_API_KEY"] = userdata.get('GEMINI_API_KEY')

# ¡CAMBIO CLAVE AQUÍ! Usamos el modelo exacto que vimos en tu lista:
llm_gemini = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.1)

# 2. Conexión a nuestra Base de Datos Vectorial
ruta_chroma = '/content/drive/MyDrive/u/cd2/chroma_db_atlas'
embeddings = HuggingFaceEmbeddings(model_name="paraphrase-multilingual-MiniLM-L12-v2")
base_vectorial = Chroma(persist_directory=ruta_chroma, embedding_function=embeddings)

# --- EVALUACIÓN DE LA RÚBRICA: TOP-K Y DISTANCIA ---
TOP_K = 3
pregunta = "¿Cuáles son las 10 competencias cardinales por las que se evalúa a los colaboradores?"

print("📊 --- EVALUACIÓN DE RECUPERACIÓN (RÚBRICA) ---")
resultados = base_vectorial.similarity_search_with_score(pregunta, k=TOP_K)

for i, (documento, distancia) in enumerate(resultados):
    print(f"\nFragmento {i+1} recuperado (Distancia L2: {distancia:.4f}):")
    print(f"📄 {documento.page_content[:200]}...")

# --- CREACIÓN DE LA CADENA RAG ---
plantilla = """
Eres un asistente corporativo experto del Banco Atlas Financiero.
Responde a la pregunta del usuario basándote ÚNICAMENTE en el siguiente contexto recuperado del manual.
Si la respuesta no está en el contexto, di "No tengo información sobre esto en las políticas actuales."
Estructura tu respuesta de forma clara (usa viñetas si es necesario).

Contexto recuperado:
{context}

Pregunta del usuario: {question}

Respuesta:
"""
prompt = PromptTemplate.from_template(plantilla)
recuperador = base_vectorial.as_retriever(search_kwargs={"k": TOP_K})

# Unimos las piezas
cadena_rag = (
    {"context": recuperador, "question": RunnablePassthrough()}
    | prompt
    | llm_gemini
    | StrOutputParser()
)

print("\n🤖 --- CALIDAD DE RESPUESTA (LLM: GEMINI) ---")
try:
    respuesta = cadena_rag.invoke(pregunta)
    print(respuesta)
    print("\n✅ ¡NIVEL 4 COMPLETADO CON ÉXITO!")
except Exception as e:
    print(f"❌ Error al consultar a Gemini: {e}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

📊 --- EVALUACIÓN DE RECUPERACIÓN (RÚBRICA) ---

Fragmento 1 recuperado (Distancia L2: 10.2676):
📄 evalúa  tanto  el  "qué"  (resultados  numéricos)  como  el  "cómo"  (alineamiento  cultural  y
metodológico).
1.2 El Modelo de Competencias Atlas (Diccionario de Competencias)
Todos los colaboradores...

Fragmento 2 recuperado (Distancia L2: 11.9939):
📄 innovación radical y asunción calculada de riesgos.
Competencia 10: Liderazgo Tecnológico
Definición: Capacidad para anticipar disrupciones en el sector financiero y aplicar soluciones
basadas en dato...

Fragmento 3 recuperado (Distancia L2: 12.4059):
📄 innovación radical y asunción calculada de riesgos.
Competencia 7: Agilidad Estratégica
Definición: Capacidad para anticipar disrupciones en el sector financiero y aplicar soluciones
basadas en datos ...

🤖 --- CALIDAD DE RESPUESTA (LLM: GEMINI) ---
No tengo información sobre esto en las políticas actuales. El contexto indica que los colaboradores son evaluados semestralmente bajo 10 compe